# Production hardening for Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

Everything to get right before a mantle workload carries real traffic. Each
section is a control you can verify, not advice you have to take on trust.

## What this notebook covers
- Credential lifecycle (short-term tokens, refresh, SigV4 (AWS Signature Version 4))
- Retries, timeouts, and the failure modes that actually occur
- Quota reality: no RPM, separate in/out TPM (tokens per minute), mostly unpublished
- Data retention posture and ZDR (zero data retention)
- Cost attribution with Projects
- Observability, including the namespace that catches people out
- Defensive output handling
- Guardrails, and why they are not a mantle parameter
- A pre-launch checklist you can run

## Self-contained, but see also
- **Auth and the three paths** →
  `../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb`
- **Governance and retention** →
  `../00-foundations/02-governance-projects-and-retention.ipynb`
- **Quotas and tiers** → `../00-foundations/03-scaling-tiers-and-latency.ipynb`

## Prerequisites
```bash
pip install -r ../requirements.txt
```

Needs openai, boto3, aws-bedrock-token-generator.

`requirements.txt` pins the exact versions this collection was tested
against. An unpinned install resolves whatever is current, which may be
untested or compromised (OWASP LLM03, Supply Chain).

### Where the helpers come from

The next cell does this:

```python
sys.path.insert(0, "../_shared")
from bedrock import ...
```

`bedrock` is **not** a package from PyPI — it is this collection's own helper
module, [`_shared/bedrock.py`](../_shared/bedrock.py). Every notebook sits one
level down, so `../_shared` puts it on the import path. It exists only to remove
repetition; the notebooks are the teaching material, and nothing in the module is
required to call Bedrock yourself.

What this notebook uses from it:

| Helper | What it does |
|---|---|
| `err` | pulls the human-readable message out of an error body, redacted |
| `list_models` | the `bedrock-mantle` model inventory for a Region |
| `parse_json_lenient` | parses the first complete JSON object out of model output, repairing truncated braces |
| `post` | signed JSON HTTP with retries. **Never raises on 4xx/5xx** — it returns `(status, body)` so a cell can *show* an error instead of stopping the notebook |
| `response_text` | assistant text from a Responses API payload |
| `resolve_runtime_id` | turns a model ID into the form Converse will accept, adding the `us.` profile prefix when one is required |
| `safe_print` | `print()` with account IDs, IAM principals and opaque service IDs redacted |

Two behaviours worth knowing before you read any output below, because several
cells depend on them:

- **`post()` and `converse()` never raise on a service error.** They return the
  status and body so a cell can *show* a 400 rather than stopping the notebook.
  Many cells here deliberately provoke an error to demonstrate a limit.
- **Anything printed from a control-plane response goes through redaction**, since
  this output is committed to a public repository.


In [1]:
import concurrent.futures as cf
import json
import random
import sys
import time
import urllib.error
import urllib.request

sys.path.insert(0, "../_shared")
from bedrock import (
    err,
    list_models,
    parse_json_lenient,
    post,
    resolve_runtime_id,
    response_text,
    safe_print,
)

REGION = "us-east-1"
MODEL = "google.gemma-4-31b"  # present in all four mantle Regions
PREFIX = "/openai/v1"
print("region:", REGION, "| model:", MODEL)

region: us-east-1 | model: google.gemma-4-31b


### Which endpoint, and the model ID for each

AWS recommends `bedrock-runtime` for new applications, and since August 2026 it
serves the OpenAI- and Anthropic-compatible APIs as well as Converse. So before
the first call, the question is which endpoint you want — and that has a
complication worth knowing about:

**the same model often carries a different ID on each endpoint.** Send a
`bedrock-mantle` ID to `bedrock-runtime` and you get *"The provided model
identifier is invalid"*, which reads like a missing model rather than a missing
translation.

The cell below asks both catalogues rather than stating an answer that will age.
`runtime_id_for()` returns `None` when a model is genuinely not on
`bedrock-runtime`, which is the honest signal for "you need mantle for this one".

In [2]:
from bedrock import endpoints_for, runtime_id_for

COVERED = [
    "anthropic.claude-fable-5",
    "anthropic.claude-haiku-4-5",
    "anthropic.claude-opus-5",
    "anthropic.claude-sonnet-5",
    "google.gemma-4-31b",
    "openai.gpt-5.6-sol",
    "openai.gpt-oss-20b",
    "qwen.qwen3-32b",
    "xai.grok-4.3",
]

print(f"{'model (as named on mantle)':38} {'on runtime as':40} endpoints")
print("-" * 96)
mantle_only = []
for model_id in COVERED:
    runtime_id = runtime_id_for(model_id, REGION)
    where = endpoints_for(model_id, REGION)
    label = ", ".join(name for name, present in where.items() if present) or "neither"
    if runtime_id is None:
        mantle_only.append(model_id)
    print(f"{model_id:38} {(runtime_id or '-- not on runtime --'):40} {label}")

renamed = [
    m for m in COVERED
    if (r := runtime_id_for(m, REGION)) is not None and r != m
]
# Cross-check the two helpers against each other. A row that prints a runtime id
# next to "mantle" only is self-contradictory, and it happened: endpoints_for()
# compared against a version-stripped catalogue key while runtime_id_for() used the
# full id, so gpt-oss showed a runtime id and "mantle". Neither helper complained.
contradictions = [
    m for m in COVERED
    if (runtime_id_for(m, REGION) is not None)
    != endpoints_for(m, REGION)["runtime"]
]
print()
if contradictions:
    print(f"!! runtime_id_for() and endpoints_for() DISAGREE for {contradictions}.")
    print("   One of them is wrong; do not trust the table above until they agree.")
print(f"=> {len(COVERED) - len(mantle_only)}/{len(COVERED)} of these are on "
      f"bedrock-runtime; {len(renamed)} under a different id.")
if mantle_only:
    print(f"   bedrock-mantle only: {mantle_only}")
    print("   For those, this notebook's endpoint is the only one that serves them.")
else:
    print("   Every model here is on both endpoints. This notebook shows the")
    print("   bedrock-mantle calls; the ids above are what you send to switch.")
print("   Region matters too: a model absent here can be present elsewhere, so")
print("   re-run this in the Region you intend to deploy in.")

model (as named on mantle)             on runtime as                            endpoints
------------------------------------------------------------------------------------------------


anthropic.claude-fable-5               us.anthropic.claude-fable-5              mantle, runtime


anthropic.claude-haiku-4-5             us.anthropic.claude-haiku-4-5-20251001-v1:0 mantle, runtime


anthropic.claude-opus-5                us.anthropic.claude-opus-5               mantle, runtime


anthropic.claude-sonnet-5              us.anthropic.claude-sonnet-5             mantle, runtime


...

openai.gpt-5.6-sol                     us.openai.gpt-5.6-sol                    mantle, runtime


openai.gpt-oss-20b                     openai.gpt-oss-20b-1:0                   mantle, runtime


qwen.qwen3-32b                         qwen.qwen3-32b-v1:0                      mantle, runtime


xai.grok-4.3                           -- not on runtime --                     mantle



=> 7/9 of these are on bedrock-runtime; 7 under a different id.
   bedrock-mantle only: ['google.gemma-4-31b', 'xai.grok-4.3']
   For those, this notebook's endpoint is the only one that serves them.
   Region matters too: a model absent here can be present elsewhere, so
   re-run this in the Region you intend to deploy in.


## 1. Credentials — mint short, refresh often, never store

Short-term Bedrock API keys are presigned SigV4 requests: they *are* IAM. They
expire within 12 hours, **cannot be refreshed**, and are Region-pinned.

Three rules:
1. Mint in-process from the ambient role. Do not put a 15-minute secret in a
   secrets manager.
2. Keep the TTL short. The token is your role until it expires.
3. Never ship long-term keys — they create a static IAM user credential.

In [3]:
import threading
from datetime import datetime, timedelta, timezone

from aws_bedrock_token_generator import provide_token


class TokenProvider:
    """Thread-safe short-term token cache with early refresh."""

    def __init__(self, region=REGION, ttl=timedelta(minutes=15), skew_s=120):
        self.region, self.ttl, self.skew_s = region, ttl, skew_s
        self._token = None
        self._expires_at = None
        self._lock = threading.Lock()
        self.mints = 0

    def get(self) -> str:
        now = datetime.now(timezone.utc)
        with self._lock:  # avoid a thundering herd
            if self._token and self._expires_at and now < self._expires_at:
                return self._token
            self._token = provide_token(region=self.region, expiry=self.ttl)
            self._expires_at = now + self.ttl - timedelta(seconds=self.skew_s)
            self.mints += 1
            return self._token


tokens = TokenProvider()
with cf.ThreadPoolExecutor(max_workers=8) as pool:
    values = list(pool.map(lambda _: tokens.get(), range(8)))
print(
    f"8 concurrent callers -> {tokens.mints} mint(s), "
    f"all identical: {len(set(values)) == 1}"
)
print("expires around:", tokens._expires_at.isoformat(timespec="seconds"))

8 concurrent callers -> 1 mint(s), all identical: True
expires around: 2026-08-25T03:05:53+00:00


In [4]:
# For roles with no need of a bearer token at all, SigV4-sign directly.
# Signing name is "bedrock". SigV4 callers do NOT need CallWithBearerToken.
import boto3
from botocore.auth import SigV4Auth
from botocore.awsrequest import AWSRequest
from botocore.httpsession import URLLib3Session

body = json.dumps({"model": MODEL, "input": "Reply OK", "max_output_tokens": 16})
creds = boto3.Session().get_credentials().get_frozen_credentials()
request = AWSRequest(
    method="POST",
    url=f"https://bedrock-mantle.{REGION}.api.aws{PREFIX}/responses",
    data=body,
    headers={"Content-Type": "application/json"},
)
SigV4Auth(creds, "bedrock", REGION).add_auth(request)
resp = URLLib3Session(timeout=60).send(request.prepare())
print("SigV4 path ->", resp.status_code)

SigV4 path -> 200


## 2. Retries, timeouts, and what actually fails

Mantle has **no RPM quota**; throttling is token-based, and most models have no
published TPM at all — capacity is internal fair-share. So `429` and `5xx` are
ordinary operating conditions, not exceptions.

Equally important: a **wrong path can stall** rather than return an error. Without
a client timeout, a retry loop turns that into a multi-minute hang.

In [5]:
# 529 is Anthropic's "overloaded", returned by the Messages API under load and
# outside the usual 5xx set. And mantle sometimes reports a SERVER fault with a 4xx
# status and the body "Internal server error", so status alone misclassifies it as
# permanent. Both are documented in ../_shared/bedrock.py; this helper is what a
# reader copies, so it has to agree with them.
TRANSIENT = {429, 500, 502, 503, 504, 529}
SERVER_FAULT_TEXT = ("internal server error", "internal failure", "internal error")


def _body(raw: bytes) -> dict:
    """Parse a response body without raising on an empty or non-JSON one.

    Always a dict. A body of `null`, `[1,2]` or `"a string"` is valid JSON but not a
    mapping, and returning it as-is made `response_text(_body(b"null"))` raise
    `AttributeError: 'NoneType' object has no attribute 'get'` -- the exact crash the
    non-dict guard in ../_shared/bedrock.py's err() was added to stop.
    """
    try:
        parsed = json.loads(raw or b"{}")
    except (ValueError, TypeError):
        # 500, matching post()'s own truncation in ../_shared/bedrock.py. At 400 the
        # two disagreed: a 400 whose body is 480 characters of plain text ending in
        # "internal server error" was retried by the library and given up on here.
        return {"raw": (raw or b"").decode("utf-8", "replace")[:500]}
    return parsed if isinstance(parsed, dict) else {"raw": parsed}


def should_retry(code: int, parsed: dict) -> bool:
    """The retry decision, as one testable predicate.

    Factored out for a reason. This used to be an inline condition, and the
    checklist at the end of the notebook then "verified" the retry policy with
    `529 in TRANSIENT` -- an expression over a literal set, which is True whatever
    the client does. A predicate can be fed a response and asked what it decides,
    so section 10 derives its ticks from a truth table over this function instead
    of from the shape of the source.

    Matching scope matters and is easy to get subtly wrong: match over the WHOLE
    serialised body, because the marker can appear in a `details` field rather than
    in `error.message`, and it can sit past any truncation limit.
    """
    if code in TRANSIENT:
        return True
    # A 4xx whose BODY reports a server fault is transient despite the status.
    # Reading the status alone gives up on a blip that succeeds on the next attempt.
    # Never widen this to all 400s: a genuine "unsupported parameter" must fail fast.
    if 400 <= code < 500:
        # default=str, like the library: a payload carrying a datetime or a set is
        # not JSON-serialisable, and a bare json.dumps raises TypeError out of a
        # function whose whole job is to return a decision.
        try:
            text = json.dumps(parsed, default=str).lower()
        except (TypeError, ValueError):
            text = str(parsed).lower()
        return any(marker in text for marker in SERVER_FAULT_TEXT)
    return False


def open_https(req, timeout: int):
    """urlopen restricted to HTTPS.

    urllib also honours file://, ftp:// and data:// . These URLs are all built
    from literals, but a client that ever takes a URL from data would let those
    schemes read local files, so the guard belongs in the helper (CWE-22).
    """
    if not req.full_url.startswith("https://"):
        raise ValueError(f"refusing non-HTTPS URL: {req.full_url[:60]}")
    # nosemgrep: dynamic-urllib-use-detected - scheme verified https above
    return urllib.request.urlopen(req, timeout=timeout)  # nosec B310  # noqa: S310


# What each call actually did. Section 10 reads this instead of asserting that the
# code looks right: an attempt count is evidence, a literal is not.
#
# Two things to change before copying this into a service. It is UNBOUNDED -- at 100
# rps that is millions of dicts a day -- so it is capped here, and in production you
# would emit to your telemetry pipeline rather than keep a list. And resilient_call
# appends BEFORE the call, so `CALL_LOG[-1]` means "my trace" only in linear code;
# section 3 drives this same client from eight threads, where the last element belongs
# to whichever thread appended most recently. resilient_call therefore RETURNS its own
# trace's index, and the cells below use that instead of [-1].
CALL_LOG: list[dict] = []
CALL_LOG_MAX = 500
# Each call's own trace, per thread. `trace` is local to resilient_call, so a cell
# cannot read it directly, and CALL_LOG[-1] belongs to whichever thread appended last.
_local = threading.local()


def last_trace() -> dict:
    """The trace recorded by THIS thread's most recent resilient_call."""
    return getattr(_local, "trace", {"path": "-", "timeout": None,
                                     "statuses": [], "sleeps": 0})


def resilient_call(path, body, *, region=REGION, headers=None, attempts=5, timeout=60):
    """Retry transient failures; fail fast on client errors; always time out."""
    url = f"https://bedrock-mantle.{region}.api.aws{path}"
    payload = json.dumps(body).encode()
    trace = {"path": path, "timeout": timeout, "statuses": [], "sleeps": 0}
    # list.append is atomic under the GIL, so no lock is needed to record; the cap
    # keeps the list bounded for a long-running process.
    if len(CALL_LOG) >= CALL_LOG_MAX:
        del CALL_LOG[:len(CALL_LOG) - CALL_LOG_MAX + 1]
    CALL_LOG.append(trace)
    _local.trace = trace
    for attempt in range(attempts):
        hdrs = {
            "Authorization": f"Bearer {tokens.get()}",
            "Content-Type": "application/json",
            **(headers or {}),
        }
        req = urllib.request.Request(url, data=payload, headers=hdrs, method="POST")
        try:
            with open_https(req, timeout=timeout) as response:
                trace["statuses"].append(response.status)
                return response.status, _body(response.read())
        except urllib.error.HTTPError as exc:
            parsed = _body(exc.read())
            trace["statuses"].append(exc.code)
            if should_retry(exc.code, parsed) and attempt < attempts - 1:
                trace["sleeps"] += 1
                time.sleep(
                    # Jitter spreads retries; not a security decision.
                    min(2**attempt, 16)
                    + random.random()  # nosec B311
                )
                continue
            return exc.code, parsed
        except Exception as exc:  # timeout, connection reset
            trace["statuses"].append(-1)
            if attempt < attempts - 1:
                trace["sleeps"] += 1
                time.sleep(
                    # Jitter spreads retries; not a security decision.
                    min(2**attempt, 16)
                    + random.random()  # nosec B311
                )
                continue
            return -1, {"error": {"message": f"{type(exc).__name__}"}}
    return -1, {"error": {"message": "retries exhausted"}}


code, data = resilient_call(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
)
print("healthy call ->", code, repr(response_text(data)[:30]))
# last_trace() is this thread's own record. CALL_LOG[-1] would be a race under the
# concurrency in section 3, where eight threads share this client.
_t = last_trace()
print(f"  attempts: {len(_t['statuses'])}, "
      f"sleeps: {_t['sleeps']}, timeout: {_t['timeout']}s")


healthy call -> 200 'OK'
  attempts: 1, sleeps: 0, timeout: 60s


In [6]:
# A permanent 400 must not be retried. Measure it: the attempt count comes from the
# call log, so this line cannot say "no retries" while the client retried.
#
# The invalid input here is `max_output_tokens` below the documented minimum of 16.
# That choice matters: an earlier version of this cell sent `top_p` to Gemma 4,
# which was a 400 when written and is a 200 now, so the cell stopped demonstrating
# anything. Pick an invariant to violate, not a per-model restriction.
started = time.perf_counter()
code, data = resilient_call(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Hi", "max_output_tokens": 8},
)
elapsed = time.perf_counter() - started
_t = last_trace()
tries = len(_t["statuses"])
sleeps = _t["sleeps"]
verdict = "no retries" if tries == 1 and sleeps == 0 else f"RETRIED {sleeps}x"
print(f"invalid param -> HTTP {code} in {elapsed:.2f}s ({verdict}, {tries} attempt(s))")
print("message:", err(data)[:90])

# And the decision itself, on both kinds of 4xx, so the policy is visible rather
# than inferred from how fast the call returned.
permanent = {"error": {"message": "Invalid 'max_output_tokens': below minimum"}}
blip = {"error": {"message": "Bad request"}, "details": "internal server error"}
print(f"\n  should_retry(400, permanent 400) -> {should_retry(400, permanent)}")
print(f"  should_retry(400, server fault) -> {should_retry(400, blip)}")
print(f"  should_retry(429, {{}})           -> {should_retry(429, {})}")
print(f"  should_retry(529, {{}})           -> {should_retry(529, {})}")


invalid param -> HTTP 400 in 0.68s (no retries, 1 attempt(s))
message: Invalid 'max_output_tokens': integer below minimum value. Expected a value >= 16, but got 

  should_retry(400, permanent 400) -> False
  should_retry(400, server fault) -> True
  should_retry(429, {})           -> True
  should_retry(529, {})           -> True


In [7]:
# The stall case, contained by a short timeout. Grok on the bare /v1 responses path
# does not answer at all.
started = time.perf_counter()
code, data = post(
    "/v1/responses",
    {"model": "xai.grok-4.3", "input": "Hi", "max_output_tokens": 16},
    region=REGION,
    attempts=1,
    timeout=20,
)
print(
    f"stalling path -> {code} in {time.perf_counter() - started:.1f}s "
    f"(bounded by the timeout, not by the server)"
)

stalling path -> -1 in 20.5s (bounded by the timeout, not by the server)


## 3. Concurrency: ramp, don't spike

A cold start from zero to peak concurrency gets shed. Step up gradually.

In [8]:
def one_call(i):
    code, _ = resilient_call(
        f"{PREFIX}/responses",
        {"model": MODEL, "input": f"Say OK ({i})", "max_output_tokens": 16},
    )
    return code


for concurrency in (1, 4, 8):
    started = time.perf_counter()
    with cf.ThreadPoolExecutor(max_workers=concurrency) as pool:
        codes = list(pool.map(one_call, range(concurrency)))
    elapsed = time.perf_counter() - started
    ok = sum(1 for c in codes if c == 200)
    print(f"concurrency {concurrency:2} -> {ok}/{concurrency} ok in {elapsed:5.2f}s")

print("\nIn production, step concurrency up over minutes and keep the backoff loop.")

concurrency  1 -> 1/1 ok in  1.02s


concurrency  4 -> 4/4 ok in  1.44s


concurrency  8 -> 8/8 ok in  1.08s

In production, step concurrency up over minutes and keep the backoff loop.


## 4. Data retention posture

Two independent controls. Get both explicit.

| Control | Scope | Default |
|---|---|---|
| `store` | per request (Responses) | **`true`** — 30-day retention |
| `data_retention.mode` | account / project / model | `inherit` |

Modes: `default`, `none` (zero data retention), `provider_data_share`, `inherit`.

In [9]:
code, retention = post("/v1/data_retention", None, region=REGION, method="GET")
print(f"account retention: HTTP {code} {retention}")

# store defaults to true — verify by omitting it.
implicit_code, implicit = post(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
    region=REGION,
)
explicit_code, explicit = post(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16, "store": False},
    region=REGION,
)
# Say what came back, and attach the retention consequence only when the response
# actually reported store=True. The parenthetical used to be a literal, so a failed
# call printed "store=None  (retained 30 days)" -- a retention claim about a request
# that never stored anything.
for label, code_, payload in (("store omitted", implicit_code, implicit),
                              ("store=False", explicit_code, explicit)):
    stored = payload.get("store") if isinstance(payload, dict) else None
    if code_ != 200:
        note = f"call failed: {err(payload)[:60]}"
    elif stored is True:
        note = "retained for 30 days"
    elif stored is False:
        note = "not retained"
    else:
        note = "the response did not report a store flag"
    print(f"{label:15} -> store={stored}  ({note})")

account retention: HTTP 200 {'mode': 'inherit'}


store omitted   -> store=True  (retained for 30 days)
store=False     -> store=False  (not retained)


In [10]:
# A retrievable response proves retention is real.
retrievable = post(
    f"{PREFIX}/responses",
    {
        "model": MODEL,
        "input": "Remember: alpha.",
        "max_output_tokens": 16,
        "store": True,
    },
    region=REGION,
)[1]
code, fetched = post(
    f"{PREFIX}/responses/{retrievable['id']}", None, region=REGION, method="GET"
)
print(f"GET stored response -> {code} (status={fetched.get('status')})")
code, _ = post(
    f"{PREFIX}/responses/{retrievable['id']}", None, region=REGION, method="DELETE"
)
print(f"DELETE it           -> {code}")

GET stored response -> 200 (status=completed)


DELETE it           -> 200


In [11]:
# Some models are gated by retention mode. Check before you debug your request.
for mid in (MODEL, "anthropic.claude-fable-5"):
    code, info = post(f"/v1/models/{mid}", None, region=REGION, method="GET")
    if code != 200:
        print(f"{mid:34} GET -> {code}")
        continue
    dr = info.get("data_retention", {})
    print(
        f"{mid:34} status={info.get('status'):12} "
        f"allowed_modes={dr.get('allowed_modes')}"
    )
    if info.get("status_reason"):
        print(f"      reason: {info['status_reason'][:100]}")

google.gemma-4-31b                 status=available    allowed_modes=['default', 'provider_data_share', 'none']


anthropic.claude-fable-5           status=unavailable  allowed_modes=['provider_data_share']
      reason: This model is not available under data retention mode 'default'.


## 5. Cost attribution with Projects

Every production workload should run under its own tagged project. Untagged usage
is impossible to allocate later.

In [12]:
code, project = post(
    "/v1/organization/projects",
    {
        "name": "hardening-demo",
        "tags": {
            "Application": "HardeningDemo",
            "Environment": "Demo",
            "Owner": "PlatformTeam",
            "CostCenter": "0000",
        },
    },
    region=REGION,
)
project_id = project.get("id")
safe_print("project:", code, project_id, "| tags:", project.get("tags"))

# The attribution header differs by API — a classic mistake.
for label, path, body, header in [
    (
        "Responses",
        f"{PREFIX}/responses",
        {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
        {"OpenAI-Project": project_id},
    ),
    (
        "ChatCompletions",
        "/v1/chat/completions",
        {
            "model": "qwen.qwen3-32b",
            "messages": [{"role": "user", "content": "Reply OK"}],
            "max_tokens": 16,
        },
        {"OpenAI-Project": project_id},
    ),
    (
        "Messages",
        "/anthropic/v1/messages",
        {
            "model": "anthropic.claude-haiku-4-5",
            "max_tokens": 16,
            "messages": [{"role": "user", "content": "Reply OK"}],
        },
        {"anthropic-workspace": project_id, "anthropic-version": "2023-06-01"},
    ),
]:
    code, _ = post(path, body, region=REGION, headers=header)
    used = [k for k in header if k != "anthropic-version"][0]
    print(f"  {label:16} via {used:22} -> {code}")

project: 200 proj_qnm7uno2... | tags: {'CostCenter': '0000', 'Application': 'HardeningDemo', 'Environment': 'Demo', 'Owner': 'PlatformTeam'}


  Responses        via OpenAI-Project         -> 200


  ChatCompletions  via OpenAI-Project         -> 200


  Messages         via anthropic-workspace    -> 200


## 6. Observability — the namespace trap

Mantle publishes to **`AWS/BedrockMantle`**, not `AWS/Bedrock`. A dashboard built
on the wrong namespace shows **zero errors during an incident**. And the only error
metric is `InferenceClientErrors` (4xx) — there is **no server-error metric**, so
503s must come from your own telemetry.

In [13]:
cw = boto3.client("cloudwatch", region_name=REGION)
# Print EVERY name, and paginate. This used to show `metrics[:6]` of 8 and of 11 and
# then assert "note the absence of any 5xx/server-error metric" -- a negative claim
# about the names it had just hidden. In AWS/Bedrock the 5 hidden names include
# `InvocationServerErrors`, the exact counterpart the comparison rests on.
SERVER_ERROR_WORDS = ("servererror", "server_error", "5xx", "servicefault",
                      "serviceerror", "throttl")
found = {}
for namespace in ("AWS/BedrockMantle", "AWS/Bedrock"):
    names = set()
    token = None
    while True:
        kwargs = {"Namespace": namespace}
        if token:
            kwargs["NextToken"] = token
        page = cw.list_metrics(**kwargs)
        names.update(m["MetricName"] for m in page.get("Metrics", []))
        token = page.get("NextToken")
        if not token:
            break
    names = sorted(names)
    found[namespace] = names
    print(f"{namespace} — {len(names)} metric(s):")
    for name in names:
        mark = "  <- server-side error metric" if any(
            w in name.lower() for w in SERVER_ERROR_WORDS) else ""
        print(f"    {name}{mark}")
    print()

# Derive the claim from the full lists.
def _server_metrics(names):
    return [n for n in names if any(w in n.lower() for w in SERVER_ERROR_WORDS)]

mantle_server = _server_metrics(found["AWS/BedrockMantle"])
bedrock_server = _server_metrics(found["AWS/Bedrock"])
print(f"server-error metrics in AWS/BedrockMantle: {mantle_server or 'none'}")
print(f"server-error metrics in AWS/Bedrock:       {bedrock_server or 'none'}")
if not mantle_server and bedrock_server:
    print("\n=> AWS/BedrockMantle publishes no server-error metric while AWS/Bedrock")
    print("   does, which is why section 6's client-side counter exists.")
elif mantle_server:
    print(f"\n=> AWS/BedrockMantle DOES publish {mantle_server}. The gotcha table at")
    print("   the end of this notebook says otherwise; fix it from this run.")
else:
    print("\n=> Neither namespace shows one here, so this comparison proves nothing")
    print("   on this account today.")
print("\nOne caveat that matters for any absence claim: list_metrics only returns")
print("metrics that have reported data recently, so 'absent' here means 'not seen in")
print("this account' rather than 'does not exist'. Read it as a prompt to check your")
print("own account, not as a service guarantee.")

AWS/BedrockMantle — 8 metric(s):
    BurnDownConsumed
    EquivalentReservationUnits
    InferenceClientErrors
    Inferences
    InputTokens
    OutputTokens
    TotalInputTokens
    TotalOutputTokens



AWS/Bedrock — 11 metric(s):
    CacheReadInputTokenCount
    CacheWriteInputTokenCount
    EstimatedTPMQuotaUsage
    InputTokenCount
    InvocationClientErrors
    InvocationLatency
    InvocationServerErrors  <- server-side error metric
    Invocations
    ModelCopy
    OutputTokenCount
    TimeToFirstToken

server-error metrics in AWS/BedrockMantle: none
server-error metrics in AWS/Bedrock:       ['InvocationServerErrors']

=> AWS/BedrockMantle publishes no server-error metric while AWS/Bedrock
   does, which is why section 6's client-side counter exists.

One caveat that matters for any absence claim: list_metrics only returns
metrics that have reported data recently, so 'absent' here means 'not seen in
this account' rather than 'does not exist'. Read it as a prompt to check your
own account, not as a service guarantee.


In [14]:
# Because 5xx is invisible server-side, record it client-side.
class CallMetrics:
    """Minimal client-side telemetry — the only place 5xx is visible."""

    def __init__(self):
        self.counts = {
            "ok": 0,
            "throttled": 0,
            "server_error": 0,
            "client_error": 0,
            "timeout": 0,
        }
        self.latencies = []

    def record(self, code, seconds):
        self.latencies.append(seconds)
        if code == 200:
            self.counts["ok"] += 1
        elif code == 429:
            self.counts["throttled"] += 1
        elif code == -1:
            self.counts["timeout"] += 1
        elif 500 <= code < 600:
            self.counts["server_error"] += 1
        else:
            self.counts["client_error"] += 1

    def report(self):
        if not self.latencies:
            return "no calls"
        ordered = sorted(self.latencies)
        # Nearest-rank for p50 as well. `ordered[n // 2]` is the UPPER median, which
        # is the same overshoot the p95 comment below identifies: n=20 over 1..20
        # gives 11 where nearest-rank p50 is 10. Fixing one and leaving the other is
        # how a percentile table ends up internally inconsistent.
        p50 = ordered[(len(ordered) * 50 + 99) // 100 - 1]
        # Nearest-rank p95 as integer arithmetic, so this needs no extra import:
        # ceil(0.95 * n) - 1. The previous expression used int(n * 0.95), which
        # overshoots whenever n * 0.95 is a whole number, so every n that is a
        # multiple of 20 reported the MAXIMUM as the 95th percentile: n=20 gave
        # index 19, value 20, where nearest-rank p95 is 19.
        p95 = ordered[(len(ordered) * 95 + 99) // 100 - 1]
        return (
            f"{self.counts} | p50={p50:.2f}s p95={p95:.2f}s " f"n={len(self.latencies)}"
        )


metrics = CallMetrics()
for i in range(6):
    started = time.perf_counter()
    code, _ = resilient_call(
        f"{PREFIX}/responses",
        {"model": MODEL, "input": f"Reply OK ({i})", "max_output_tokens": 16},
        headers={"OpenAI-Project": project_id},
    )
    metrics.record(code, time.perf_counter() - started)
print(metrics.report())

{'ok': 6, 'throttled': 0, 'server_error': 0, 'client_error': 0, 'timeout': 0} | p50=0.90s p95=0.94s n=6


**CloudTrail:** mantle inference is a **data event** — off by default, and billed
extra when enabled. Management events (creating projects) appear normally. Note
that short-term key *generation* is client-side and never logged.

## 7. Defensive output handling

HTTP 200 does not mean you got what you asked for. Three real cases:

1. **Truncation** — `finish_reason="length"` with empty content.
2. **Trailing characters** after valid JSON, even in strict mode.
3. **Ignored constraints** — a forced tool choice that returns prose.

In [15]:
SCHEMA = {
    "type": "object",
    "properties": {"language": {"type": "string"}, "typed": {"type": "boolean"}},
    "required": ["language", "typed"],
    "additionalProperties": False,
}


def safe_structured(
    model, prompt, schema, *, prefix=PREFIX, max_output_tokens=600, attempts=3
):
    """Structured output with truncation detection, lenient parsing and retries."""
    budget = max_output_tokens
    for attempt in range(attempts):
        code, data = post(
            f"{prefix}/responses",
            {
                "model": model,
                "input": prompt,
                "max_output_tokens": budget,
                "text": {
                    "format": {
                        "type": "json_schema",
                        "name": "out",
                        "schema": schema,
                        "strict": True,
                    }
                },
                "store": False,
            },
            region=REGION,
        )
        if code != 200:
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        text = response_text(data)
        if not text.strip():
            budget *= 3  # truncated before emitting anything
            print(f"   attempt {attempt + 1}: empty output, raising budget to {budget}")
            continue
        try:
            parsed = parse_json_lenient(text)  # tolerates trailing characters
        except ValueError as exc:
            print(f"   attempt {attempt + 1}: unparseable ({exc})")
            continue
        missing = set(schema["required"]) - set(parsed)
        if missing:
            print(f"   attempt {attempt + 1}: missing keys {missing}")
            continue
        return parsed
    raise RuntimeError("no valid structured output after retries")


print("structured with guards:")
print("  ", safe_structured(MODEL, "Describe the Go programming language.", SCHEMA))

structured with guards:


   {'language': 'Go', 'typed': True}


In [16]:
# Show the trailing-character problem that motivates lenient parsing.
raw_samples = [
    '{"language":"Go","typed":true}',
    '{"language":"Go","typed":true}\n}',
    '{"language":"Go","typed":true}\nextra text',
]
for sample in raw_samples:
    try:
        json.loads(sample)
        verdict = "json.loads OK"
    except json.JSONDecodeError:
        verdict = "json.loads FAILS"
    print(f"  {verdict:18} | lenient -> {parse_json_lenient(sample)} | {sample!r}")

  json.loads OK      | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}'
  json.loads FAILS   | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}\n}'
  json.loads FAILS   | lenient -> {'language': 'Go', 'typed': True} | '{"language":"Go","typed":true}\nextra text'


## 8. Region and model availability as a pre-flight check

Verify at startup that every model you depend on exists in your Region. Failing
here is much better than failing on first traffic.

In [17]:
REQUIRED = [MODEL, "qwen.qwen3-32b", "anthropic.claude-haiku-4-5", "openai.gpt-5.6-sol"]


def preflight(required, region=REGION):
    """Fail fast at startup if a dependency is missing from this Region."""
    available = set(list_models(region))
    missing = [m for m in required if m not in available]
    return {
        "region": region,
        "ok": not missing,
        "missing": missing,
        "available_count": len(available),
    }


for region in ("us-east-1", "eu-central-1"):
    result = preflight(REQUIRED, region)
    status = "PASS" if result["ok"] else "FAIL"
    print(
        f"{region:14} {status}  ({result['available_count']} models) "
        f"missing={result['missing']}"
    )

us-east-1      PASS  (55 models) missing=[]


eu-central-1   FAIL  (33 models) missing=['anthropic.claude-haiku-4-5', 'openai.gpt-5.6-sol']


## 9. A production client, assembled

Every control from above in one place.

In [18]:
def tool_use_result(data: dict, schema: dict) -> dict:
    """Pull the object out of a forced tool call, or say why you cannot.

    A named function rather than a block inside ask(), so the cell below can hand it
    a payload and show what it does. Inline, the guard could only be described --
    and the description said "ask() raises rather than returning it" beside a
    computation that never called ask().

    `or {}` alone returned an empty object as though the schema had been satisfied,
    so the same required-key check safe_structured() applies in section 7 is applied
    here.
    """
    for block in data.get("content") or []:
        if block.get("type") == "tool_use":
            got = block.get("input") or {}
            missing = set(schema.get("required") or []) - set(got)
            if missing:
                raise RuntimeError(
                    f"forced tool returned an object missing {sorted(missing)} — "
                    f"raise max_output_tokens"
                )
            return got
    raise RuntimeError(
        "forced tool returned no tool_use block — retry or raise max_output_tokens"
    )


class ProductionClient:
    """Hardened bedrock-mantle client.

    Routing is the part that used to be wrong here. An earlier version computed the
    right path *prefix* per model and then always POSTed `{prefix}/responses`, so it
    400ed for every Claude model and every Chat-Completions-only family — i.e. for
    most of the catalogue. Resolve the **surface** as well as the prefix.

    Sampling is handled by dropping what the service names in a 400 rather than by
    carrying a per-model table. Those tables have gone stale twice in this
    collection's lifetime; the error message has not moved.
    """

    CHAT_ONLY = ("qwen.", "deepseek.", "zai.", "minimax.", "moonshotai.",
                 "mistral.", "nvidia.", "writer.", "openai.gpt-oss-safeguard",
                 "google.gemma-3")
    COMPLETION_TOKENS = ("openai.gpt-5.6",)
    TIERED = ("openai.gpt-oss", "google.gemma-", "xai.", "qwen.", "deepseek.",
              "zai.", "minimax.", "moonshotai.", "mistral.", "nvidia.", "writer.")
    TUNABLE = ("temperature", "top_p", "service_tier")

    def __init__(self, model, region=REGION, project=None, tier="default"):
        self.model, self.region, self.project = model, region, project
        self.tokens = TokenProvider(region=region)
        self.metrics = CallMetrics()
        if model.startswith("anthropic."):
            self.prefix, self.surface = "/anthropic/v1", "messages"
        elif model.startswith(self.CHAT_ONLY):
            self.prefix = "/openai/v1" if model.startswith(
                ("google.gemma-4", "openai.gpt-5", "xai.")) else "/v1"
            self.surface = "chat"
        else:
            self.prefix = "/openai/v1" if model.startswith(
                ("google.gemma-4", "openai.gpt-5", "xai.")) else "/v1"
            self.surface = "responses"
        self.tier = (tier if (tier == "default" or model.startswith(self.TIERED))
                     else "default")

    def _budget_field(self):
        if self.surface == "responses":
            return "max_output_tokens"
        if self.surface == "chat" and self.model.startswith(self.COMPLETION_TOKENS):
            return "max_completion_tokens"
        return "max_tokens"

    def _build(self, prompt, budget, schema, sampling):
        body = {"model": self.model, self._budget_field(): max(16, budget),
                **sampling}
        if self.surface == "messages":
            body["messages"] = [{"role": "user", "content": prompt}]
            # Claude has no response_format and no text.format: output_config is
            # rejected on mantle, so a forced tool is the route (established in
            # 01-choosing-a-model-and-api §5). This branch used to ignore `schema`
            # and return, so ask() ran its schema post-processing on ordinary prose
            # and parse_json_lenient raised -- or, worse, accepted a JSON-ish answer
            # as though the schema had been enforced.
            if schema:
                body["tools"] = [
                    {
                        "name": "out",
                        "description": "Return the result.",
                        "input_schema": schema,
                    }
                ]
                body["tool_choice"] = {"type": "tool", "name": "out"}
            return f"{self.prefix}/messages", body
        # `self.tier` is None only after ask() has walked it down. Sending
        # "default" is legal and is the first-attempt behaviour; a model that
        # refuses the PARAMETER rather than the value is recovered by ask() setting
        # tier to None, which omits the field entirely (see 02 §7). The comment here
        # used to claim this line omitted "default", which it does not -- "default"
        # is truthy.
        if self.tier is not None:
            body["service_tier"] = self.tier
        if self.surface == "chat":
            body["messages"] = [{"role": "user", "content": prompt}]
            if schema:
                body["response_format"] = {
                    "type": "json_schema",
                    "json_schema": {"name": "out", "strict": True, "schema": schema},
                }
            return f"{self.prefix}/chat/completions", body
        body["input"] = prompt
        body["store"] = False  # explicit: no 30-day retention
        if schema:
            body["text"] = {"format": {"type": "json_schema", "name": "out",
                                       "schema": schema, "strict": True}}
        return f"{self.prefix}/responses", body

    @staticmethod
    def _refused_param(message):
        """Which tunable the service just named, if any. See 02-migrating §7."""
        import re

        quoted = re.findall(r"[\'`\"]([a-z_]+)[\'`\"]", message or "")
        for name in quoted:
            if name in ProductionClient.TUNABLE:
                return name
        if "API" not in (message or ""):
            for name in ProductionClient.TUNABLE:
                if name in (message or ""):
                    return name
        return None

    def _extract(self, data):
        if self.surface == "messages":
            return "".join(b.get("text", "") for b in data.get("content", [])
                           if b.get("type") == "text")
        if self.surface == "chat":
            return (data.get("choices") or [{}])[0].get("message", {}).get(
                "content") or ""
        return response_text(data)

    def ask(self, prompt, *, max_output_tokens=512, temperature=None, top_p=None,
            schema=None):
        sampling = {}
        if temperature is not None:
            sampling["temperature"] = temperature
        if top_p is not None:
            sampling["top_p"] = top_p

        headers = {}
        if self.project:
            key = ("anthropic-workspace" if self.surface == "messages"
                   else "OpenAI-Project")
            headers[key] = self.project
        if self.surface == "messages":
            headers["anthropic-version"] = "2023-06-01"

        # One attempt, plus one for each recovery step that could be needed. That is
        # NOT len(TUNABLE) + 1: service_tier has TWO steps -- downgrade "flex" to
        # "default", then omit the field -- so a model refusing service_tier as a
        # parameter AND temperature AND top_p needs five attempts. With four, the
        # fully stripped request, the one that would have succeeded, was never sent,
        # and the client raised instead. The committed run in 02 §7 already consumes
        # all four: "dropped service_tier=default, temperature, top_p and retried".
        dropped: list[str] = []
        for _ in range(len(self.TUNABLE) + 2):
            path, body = self._build(prompt, max_output_tokens, schema, sampling)
            started = time.perf_counter()
            code, data = post(path, body, region=self.region,
                              headers=headers or None, timeout=90)
            self.metrics.record(code, time.perf_counter() - started)
            if code == 200:
                # A forced tool returns the object in the tool_use block, already
                # structured; there is no JSON text to parse.
                if schema and self.surface == "messages":
                    return tool_use_result(data, schema)
                text = self._extract(data)
                if schema:
                    if not text.strip():
                        raise RuntimeError("empty output — raise max_output_tokens")
                    return parse_json_lenient(text)
                return text
            # Full message, not the 160-character display form: the parameter name
            # can sit past the truncation point (see 02-migrating-from-openai §7).
            refused = self._refused_param(err(data, limit=2000))
            if refused == "service_tier":
                if self.tier not in (None, "default"):
                    self.tier = "default"
                    dropped.append("service_tier=default")
                    continue
                if self.tier == "default":
                    self.tier = None
                    dropped.append("service_tier omitted")
                    continue
            if refused in sampling:
                sampling.pop(refused)
                dropped.append(refused)
                continue
            raise RuntimeError(f"HTTP {code}: {err(data)}")
        raise RuntimeError(
            f"still refused after dropping {dropped or 'nothing'} — "
            f"the last response was HTTP {code}: {err(data)}"
        )


bot = ProductionClient(MODEL, project=project_id, tier="flex")

Exercise it, then read the metrics it recorded.

In [19]:
print("plain     :", bot.ask("Name one benefit of fair-share scheduling.")[:120])
print(
    "structured:",
    bot.ask("Describe the Rust language.", schema=SCHEMA, max_output_tokens=400),
)
print("metrics   :", bot.metrics.report())

# The routing fix, exercised: the same class against a Chat-Completions-only model
# and a Messages-only model. Both 400ed in the previous version of this notebook.
for other in ("qwen.qwen3-32b", "anthropic.claude-haiku-4-5"):
    probe = ProductionClient(other, project=project_id, tier="flex")
    answer = probe.ask("Name one benefit of queues. One sentence.",
                       max_output_tokens=120, temperature=0.7, top_p=0.95)
    print(f"{other:28} [{probe.surface}] {' '.join(answer.split())[:60]}")

# And the Claude forced-tool route, which nothing here exercised until now. That
# branch exists because Claude has no response_format and no text.format, so a
# schema has to become a tool with tool_choice pinned to it. It was added to fix a
# real bug -- the messages branch used to ignore `schema` entirely and ask() then ran
# JSON post-processing over ordinary prose -- and an unexercised fix for a real bug
# is indistinguishable from no fix. Every call above passed schema=None on the
# messages surface, so the branch, its RuntimeError and its required-key check had
# never run.
print()
for claude in ("anthropic.claude-haiku-4-5", "anthropic.claude-opus-5"):
    probe = ProductionClient(claude, project=project_id)
    try:
        got = probe.ask("Describe the Rust programming language.",
                        schema=SCHEMA, max_output_tokens=600)
        keys_ok = set(SCHEMA["required"]) <= set(got)
        print(f"{claude:28} [{probe.surface}, forced tool] {got}  "
              f"required keys present: {keys_ok}")
    except RuntimeError as exc:
        # The failure path is a result too: it says the branch ran and rejected an
        # answer that did not satisfy the schema, which is what it is for.
        print(f"{claude:28} [{probe.surface}, forced tool] raised: {exc}")

# The guard itself, exercised rather than described. `tool_use_result` is the function
# ask() calls, so handing it a payload IS running the guard. The previous version
# computed the missing keys from a literal and then printed a claim about ask() that it
# never called -- a verdict beside a probe that was not run, which is exactly the
# defect this notebook is about.
print()
for label, payload in [
    ("complete", {"content": [{"type": "tool_use", "name": "out",
                               "input": {"language": "Go", "typed": True}}]}),
    ("missing a required key", {"content": [{"type": "tool_use", "name": "out",
                                            "input": {"language": "Go"}}]}),
    ("empty tool input", {"content": [{"type": "tool_use", "name": "out",
                                       "input": {}}]}),
    ("no tool_use block at all", {"content": [{"type": "text", "text": "Go is..."}]}),
]:
    try:
        print(f"  {label:24} -> returned {tool_use_result(payload, SCHEMA)}")
    except RuntimeError as exc:
        print(f"  {label:24} -> raised: {exc}")


plain     : One major benefit of fair-share scheduling is that it **prevents a single user or group from monopolizing system resourc


structured: {'language': 'Rust', 'typed': True}
metrics   : {'ok': 2, 'throttled': 0, 'server_error': 0, 'client_error': 0, 'timeout': 0} | p50=1.01s p95=1.65s n=2


qwen.qwen3-32b               [chat] Queues ensure orderly processing by managing tasks or reques


anthropic.claude-haiku-4-5   [messages] Queues ensure that tasks or requests are processed in the or



anthropic.claude-haiku-4-5   [messages, forced tool] {'language': 'markdown', 'typed': True}  required keys present: True


anthropic.claude-opus-5      [messages, forced tool] {'language': 'Rust', 'typed': True}  required keys present: True

  complete                 -> returned {'language': 'Go', 'typed': True}
  missing a required key   -> raised: forced tool returned an object missing ['typed'] — raise max_output_tokens
  empty tool input         -> raised: forced tool returned an object missing ['language', 'typed'] — raise max_output_tokens
  no tool_use block at all -> raised: forced tool returned no tool_use block — retry or raise max_output_tokens


## 9b. Guardrails — and the surfaces that accept the header without using it

Amazon Bedrock Guardrails is the service's content-safety control: denied topics,
content filters, word filters, PII redaction, contextual grounding. AWS documents
it as a `bedrock-runtime` feature, and it is — but *"available on bedrock-runtime"*
is not granular enough to build on, because **the five APIs on that endpoint do not
all honour it**.

The cells below measure every attachment point, and the last column says which cell
does it — because four of these rows were asserted with no probe anywhere in the
notebook while this paragraph claimed otherwise. Read the results the cells print,
not this table: the table is the index.

| How you attach it | Result | Measured by |
|---|---|---|
| `guardrailConfig` on **Converse** | **enforced** — `stopReason=guardrail_intervened` | §9b Converse cell |
| **`ApplyGuardrail`** called directly | **enforced** — returns `GUARDRAIL_INTERVENED` | §9b ApplyGuardrail cell |
| `guardrailIdentifier` on **InvokeModel** | **enforced** | §9b four-row cell |
| `X-Amzn-Bedrock-Guardrail*` headers on runtime **Chat Completions** | **enforced** | §9b six-surface cell |
| `X-Amzn-Bedrock-Guardrail*` headers on runtime **Messages** | **enforced** | §9b six-surface cell |
| `X-Amzn-Bedrock-Guardrail*` headers on runtime **Responses** | **200 — silently ignored** | §9b six-surface cell |
| `X-Amzn-Bedrock-Guardrail*` headers on any **`bedrock-mantle`** surface | **200 — silently ignored** | §9b six-surface cell |
| `guardrailConfig` in the **body** of Chat Completions, either endpoint | **200 — silently ignored** | §9b four-row cell |
| `guardrailConfig` in the **body** of Responses | 400 — safe | §9b four-row cell |
| `guardrailConfig` in the **body** of Messages | 400 — safe | §9b four-row cell |

**The silently-ignored rows are the dangerous ones.** The request succeeds, nothing
in the response says the guardrail was skipped, and a team that sets the header and
sees HTTP 200 believes it has protection it does not have. The rows that return a
400 are *safer*, because they fail at the first call in development.

### How to test this yourself, and why one test is not enough

A guardrail block and a model declining on its own look identical from outside. Ask
a model for investment advice and it may refuse for its own reasons — so a probe
built on a topic models dislike cannot distinguish "the guardrail worked" from "the
model was cautious". This section therefore uses two signals:

1. **A DENY topic no model has any reason to refuse** — tulip cultivation — with a
   no-header control proving the model answers it happily.
2. **A guardrail identifier that does not exist.** A surface that enforces
   guardrails must reject it; a surface that ignores the header returns 200. This
   signal does not depend on model behaviour at all.

When the two agree, the row is trustworthy. That matters here specifically: an
earlier version of this notebook probed only the Responses surface and generalised
its result to *"bedrock-runtime's OpenAI APIs"*. The result was right; the
generalisation was not.

In [20]:
import boto3

control = boto3.client("bedrock", region_name=REGION)
runtime = boto3.client("bedrock-runtime", region_name=REGION)

# The DENY topic is deliberately banal. A guardrail that blocks investment advice
# cannot be told apart from a model that declines to give investment advice, and
# that ambiguity is how a wrong conclusion gets published. No model refuses to
# discuss tulips, so a refusal here can only be the guardrail.
GUARDRAIL_NAME = "bedrock-samples-hardening-demo"

guardrail = control.create_guardrail(
    name=GUARDRAIL_NAME,
    description="Throwaway guardrail for the per-model Bedrock samples. Safe to delete.",
    topicPolicyConfig={
        "topicsConfig": [
            {
                "name": "Tulips",
                "definition": "Any discussion of tulips, tulip bulbs, or tulip "
                              "cultivation.",
                "examples": ["Tell me about tulips."],
                "type": "DENY",
            }
        ]
    },
    contentPolicyConfig={
        "filtersConfig": [
            {"type": "VIOLENCE", "inputStrength": "HIGH", "outputStrength": "HIGH"}
        ]
    },
    blockedInputMessaging="BLOCKED_BY_GUARDRAIL (input).",
    blockedOutputsMessaging="BLOCKED_BY_GUARDRAIL (output).",
)
GUARDRAIL_ID, GUARDRAIL_VERSION = guardrail["guardrailId"], guardrail["version"]
safe_print("created guardrail:", GUARDRAIL_ID, "version", GUARDRAIL_VERSION)

# A new guardrail takes a moment to become READY. Polling beats sleeping: the
# wait is short on a good day and this does not fail on a slow one.
for _ in range(25):
    status = control.get_guardrail(
        guardrailIdentifier=GUARDRAIL_ID, guardrailVersion=GUARDRAIL_VERSION
    )["status"]
    if status == "READY":
        break
    time.sleep(2)
print("status:", status)

BLOCK_MARK = "BLOCKED_BY_GUARDRAIL"
DENIED = "In two sentences, describe how tulips are cultivated."
ALLOWED = "In two sentences, describe how potatoes are cultivated."

created guardrail: 5murea8qm64o version DRAFT


status: READY


In [21]:
# ApplyGuardrail evaluates text with no model and no inference cost, so it works
# as a pre-filter in front of ANY endpoint or API -- including the surfaces further
# down that ignore the header. If you need guardrails on one of those, this is the
# answer.
print("ApplyGuardrail — a pre-filter you can run before any endpoint or API")
print("-" * 72)
for text in (DENIED, ALLOWED):
    verdict = runtime.apply_guardrail(
        guardrailIdentifier=GUARDRAIL_ID,
        guardrailVersion=GUARDRAIL_VERSION,
        source="INPUT",
        content=[{"text": {"text": text}}],
    )
    topics = [
        t["name"]
        for assessment in verdict.get("assessments", [])
        for t in assessment.get("topicPolicy", {}).get("topics", [])
    ]
    print(f"  {verdict['action']:22} topics={str(topics):24} {text[:38]}")

print()
print("=> action=GUARDRAIL_INTERVENED means do not send it. Run the same call with")
print("   source='OUTPUT' on the model's reply to screen what you return.")
print("   Two calls per turn is the cost of guardrailing a surface that will not")
print("   guardrail itself.")

ApplyGuardrail — a pre-filter you can run before any endpoint or API
------------------------------------------------------------------------


  GUARDRAIL_INTERVENED   topics=['Tulips']               In two sentences, describe how tulips 


  NONE                   topics=[]                       In two sentences, describe how potatoe

=> action=GUARDRAIL_INTERVENED means do not send it. Run the same call with
   source='OUTPUT' on the model's reply to screen what you return.
   Two calls per turn is the cost of guardrailing a surface that will not
   guardrail itself.


In [22]:
# Shape 1: guardrailConfig on Converse. Works across providers, and the stop
# reason names what happened.
print("Converse with guardrailConfig")
print("-" * 72)
for model in ("amazon.nova-micro-v1", "anthropic.claude-sonnet-5"):
    try:
        reply = runtime.converse(
            modelId=resolve_runtime_id(model, REGION),
            messages=[{"role": "user", "content": [{"text": DENIED}]}],
            inferenceConfig={"maxTokens": 200},
            guardrailConfig={
                "guardrailIdentifier": GUARDRAIL_ID,
                "guardrailVersion": GUARDRAIL_VERSION,
            },
        )
        text = "".join(
            b.get("text", "") for b in reply["output"]["message"]["content"]
        )
        print(f"  {model:28} stop={reply['stopReason']:22} "
              f"blocked={BLOCK_MARK in text}")
    except Exception as exc:  # noqa: BLE001 - report, do not stop the notebook
        print(f"  {model:28} {type(exc).__name__}: {str(exc)[-60:]}")

# Converse also VALIDATES the identifier, which is the behaviour to expect from a
# surface that really applies it.
try:
    runtime.converse(
        modelId=resolve_runtime_id("amazon.nova-micro-v1", REGION),
        messages=[{"role": "user", "content": [{"text": "Hi"}]}],
        inferenceConfig={"maxTokens": 16},
        guardrailConfig={"guardrailIdentifier": "gr-doesnotexist000",
                         "guardrailVersion": "1"},
    )
    print("  bogus identifier             accepted (200) <- would be a bad sign")
except Exception as exc:  # noqa: BLE001
    print(f"  bogus identifier             rejected: {type(exc).__name__}")

Converse with guardrailConfig
------------------------------------------------------------------------


  amazon.nova-micro-v1         stop=guardrail_intervened   blocked=True


  anthropic.claude-sonnet-5    stop=guardrail_intervened   blocked=True


  bogus identifier             rejected: ValidationException


### The six OpenAI- and Anthropic-shaped surfaces, measured

Same guardrail, same question, six surfaces. Each row reports both signals:
whether the banal denied topic was actually blocked, and whether a nonexistent
guardrail ID was rejected. The final column flags any row where the two disagree —
if that happens, do not trust the row, re-run it.

In [23]:
RUNTIME_HOST = f"https://bedrock-runtime.{REGION}.amazonaws.com"
MANTLE_HOST = f"https://bedrock-mantle.{REGION}.api.aws"

SURFACES = [
    # (label, host, path, response shape, body, extra headers)
    ("runtime Chat Completions", RUNTIME_HOST, "/openai/v1/chat/completions", "chat",
     {"model": "openai.gpt-oss-20b-1:0", "max_completion_tokens": 2000}, {}),
    ("runtime Responses", RUNTIME_HOST, "/openai/v1/responses", "responses",
     {"model": "us.openai.gpt-5.6-sol", "max_output_tokens": 2000}, {}),
    ("runtime Messages", RUNTIME_HOST, "/anthropic/v1/messages", "messages",
     {"model": "us.anthropic.claude-opus-5", "max_tokens": 300},
     {"anthropic-version": "2023-06-01"}),
    # 2000, not 300: gpt-oss emits a <reasoning> block before the answer, and a
    # tight budget leaves the control with too little text to count as answered.
    ("mantle Chat Completions", MANTLE_HOST, "/v1/chat/completions", "chat",
     {"model": "openai.gpt-oss-20b", "max_tokens": 2000}, {}),
    ("mantle Responses", MANTLE_HOST, "/openai/v1/responses", "responses",
     {"model": "openai.gpt-5.6-sol", "max_output_tokens": 2000}, {}),
    ("mantle Messages", MANTLE_HOST, "/anthropic/v1/messages", "messages",
     {"model": "anthropic.claude-opus-5", "max_tokens": 300},
     {"anthropic-version": "2023-06-01"}),
]

GOOD_HEADERS = {
    "X-Amzn-Bedrock-GuardrailIdentifier": GUARDRAIL_ID,
    "X-Amzn-Bedrock-GuardrailVersion": GUARDRAIL_VERSION,
}
BOGUS_HEADERS = {
    "X-Amzn-Bedrock-GuardrailIdentifier": "gr-doesnotexist000",
    "X-Amzn-Bedrock-GuardrailVersion": "1",
}


def send(base, path, shape, body, headers):
    """POST with a bearer token; return (status, assistant text)."""
    request_body = dict(body)
    if shape == "responses":
        request_body["input"] = DENIED
    else:
        request_body["messages"] = [{"role": "user", "content": DENIED}]
    hdrs = {
        "Authorization": f"Bearer {provide_token(region=REGION)}",
        "Content-Type": "application/json",
        **headers,
    }
    request = urllib.request.Request(
        base + path, data=json.dumps(request_body).encode(), headers=hdrs,
        method="POST",
    )
    try:
        # URL is a literal host constant plus a literal path, so the scheme is
        # https by construction and urllib's other schemes are unreachable.
        # nosemgrep: dynamic-urllib-use-detected
        with urllib.request.urlopen(request, timeout=180) as reply:  # nosec B310  # noqa: S310
            status, raw = reply.status, reply.read().decode("utf-8", "replace")
    except urllib.error.HTTPError as exc:
        return exc.code, exc.read().decode("utf-8", "replace")[:120]
    except Exception as exc:  # noqa: BLE001
        return -1, f"{type(exc).__name__}"

    payload = json.loads(raw) if raw.strip() else {}
    if shape == "responses":
        return status, response_text(payload)
    if shape == "messages":
        return status, "".join(
            b.get("text", "") for b in (payload.get("content") or [])
            if isinstance(b, dict)
        )
    message = (payload.get("choices") or [{}])[0].get("message") or {}
    return status, message.get("content") or ""


print(f"{'surface':26} {'control':>8} {'header':>8} {'bogus id':>10} {'verdict':>22} agree")
print("-" * 92)
rows = []
for label, base, path, shape, body, extra in SURFACES:
    # Control: no guardrail. Proves the model answers, so a later refusal is
    # attributable to the guardrail rather than to the model's own caution.
    control_status, control_text = send(base, path, shape, body, extra)
    # Keep the control's STATUS too. Discarding it made a FAILED control read as
    # answered: send() returns the error body on an HTTPError, and an error body is
    # comfortably longer than 40 characters, so `answered` was True, the "control
    # failed" warning never printed, and the row was published as trustworthy.
    answered = control_status == 200 and len(control_text.strip()) > 40

    # Keep the STATUS. Discarding it collapsed two opposite outcomes into one: a
    # surface that REFUSES the guardrail header with a 400 has no block marker in the
    # body either, so it printed "IGNORED (accepts, no-ops)" -- the header rejected,
    # reported as accepted. In a security table that is the wrong direction to err.
    guarded_status, guarded_text = send(
        base, path, shape, body, {**extra, **GOOD_HEADERS}
    )
    blocked = guarded_status == 200 and BLOCK_MARK in guarded_text
    accepts_header = guarded_status == 200

    bogus_status, _ = send(base, path, shape, body, {**extra, **BOGUS_HEADERS})
    # A 4xx, not merely "not 200": send() returns -1 for a timeout or connection
    # reset, and `!= 200` counted that as "the surface validated the guardrail ID" --
    # a network blip reported as a security control working. And not 429 either: a
    # throttle is not validation, which is the same class of error one narrowing
    # short of.
    rejects_bogus = 400 <= bogus_status < 500 and bogus_status != 429

    if not accepts_header:
        verdict = f"HEADER REFUSED ({guarded_status})"
    elif blocked:
        verdict = "ENFORCED"
    else:
        verdict = "ACCEPTED, NOT ENFORCED"
    # The two signals only measure the same thing when the header was accepted.
    agree = (blocked == rejects_bogus) if accepts_header else True
    rows.append({"surface": label, "enforced": blocked, "agree": agree,
                 "answered": answered, "accepts": accepts_header,
                 "control_status": control_status, "bogus_status": bogus_status})
    print(f"{label:26} {str(answered):>8} {str(blocked):>8} "
          f"{('reject' if rejects_bogus else 'accept'):>10} {verdict:>22} {agree}")

print()
enforced = [r["surface"] for r in rows if r["enforced"]]
ignored = [r["surface"] for r in rows
           if not r["enforced"] and r.get("accepts", True)]
refused = [r["surface"] for r in rows if not r.get("accepts", True)]
disagreed = [r["surface"] for r in rows if not r["agree"]]
unanswered = [r["surface"] for r in rows if not r["answered"]]

if unanswered:
    print(f"!! control failed on {unanswered}: no usable answer to the benign")
    print("   question, so those rows prove nothing. Fix the control first.")
    for r in rows:
        if not r["answered"]:
            print(f"     {r['surface']}: control returned HTTP {r['control_status']}")
inconclusive = [r["surface"] for r in rows if r["bogus_status"] == -1]
if inconclusive:
    print(f"!! the bogus-identifier signal timed out on {inconclusive}, so the")
    print("   two-signal agreement check below is inconclusive for those rows.")
print(f"=> guardrail header ENFORCED on: {enforced or 'none'}")
print(f"=> guardrail header ACCEPTED BUT NOT ENFORCED on: {ignored or 'none'}")
print(f"=> guardrail header REFUSED outright on         : {refused or 'none'}")
if disagreed:
    print(f"!! the two signals disagree on {disagreed} — re-run before trusting it.")
else:
    print("   Both signals agreed on every row: the surfaces that blocked the topic")
    print("   are exactly the surfaces that rejected a nonexistent guardrail ID.")
print()
print("   The operational rule: attach a guardrail, then VERIFY it on the exact")
print("   surface you ship. 'Guardrails are supported on this endpoint' is true and")
print("   still leaves you unprotected on some of its APIs. Where the header is")
print("   ignored, use ApplyGuardrail explicitly or move the call to Converse.")

surface                     control   header   bogus id                verdict agree
--------------------------------------------------------------------------------------------


runtime Chat Completions       True     True     reject               ENFORCED True


runtime Responses              True    False     accept ACCEPTED, NOT ENFORCED True


runtime Messages               True     True     reject               ENFORCED True


mantle Chat Completions        True    False     accept ACCEPTED, NOT ENFORCED True


mantle Responses               True    False     accept ACCEPTED, NOT ENFORCED True


mantle Messages                True    False     accept ACCEPTED, NOT ENFORCED True

=> guardrail header ENFORCED on: ['runtime Chat Completions', 'runtime Messages']
=> guardrail header ACCEPTED BUT NOT ENFORCED on: ['runtime Responses', 'mantle Chat Completions', 'mantle Responses', 'mantle Messages']
=> guardrail header REFUSED outright on         : none
   Both signals agreed on every row: the surfaces that blocked the topic
   are exactly the surfaces that rejected a nonexistent guardrail ID.

   The operational rule: attach a guardrail, then VERIFY it on the exact
   surface you ship. 'Guardrails are supported on this endpoint' is true and
   still leaves you unprotected on some of its APIs. Where the header is
   ignored, use ApplyGuardrail explicitly or move the call to Converse.


In [24]:
# The four rows in the table above that nothing measured. The section promises "the
# cells below measure every attachment point", and until now four verdicts were
# asserted: guardrailIdentifier on InvokeModel, and guardrailConfig in the BODY of
# each of the three OpenAI/Anthropic-shaped APIs. The InvokeModel row is the
# load-bearing one, because it is the fallback this section tells you to use.
print("guardrailConfig in the request BODY (not a header):")
BODY_SURFACES = [
    ("runtime Chat Completions", RUNTIME_HOST, "/openai/v1/chat/completions", "chat",
     {"model": "openai.gpt-oss-20b-1:0", "max_completion_tokens": 2000}, {}),
    ("runtime Responses", RUNTIME_HOST, "/openai/v1/responses", "responses",
     {"model": "us.openai.gpt-5.6-sol", "max_output_tokens": 2000}, {}),
    ("runtime Messages", RUNTIME_HOST, "/anthropic/v1/messages", "messages",
     {"model": "us.anthropic.claude-opus-5", "max_tokens": 300},
     {"anthropic-version": "2023-06-01"}),
    ("mantle Chat Completions", MANTLE_HOST, "/v1/chat/completions", "chat",
     {"model": "openai.gpt-oss-20b", "max_tokens": 2000}, {}),
]
GUARDRAIL_BODY = {"guardrailConfig": {"guardrailIdentifier": GUARDRAIL_ID,
                                      "guardrailVersion": GUARDRAIL_VERSION}}
body_rows = []
for label, base, path_, shape, body, extra in BODY_SURFACES:
    status, text = send(base, path_, shape, {**body, **GUARDRAIL_BODY}, extra)
    blocked = status == 200 and BLOCK_MARK in text
    if status != 200:
        verdict = f"{status} REFUSED — safe, fails in development"
    elif blocked:
        verdict = "200 and ENFORCED"
    else:
        verdict = "200 and SILENTLY IGNORED"
    body_rows.append({"surface": label, "status": status, "enforced": blocked})
    print(f"  {label:26} {verdict}")
    if status != 200:
        print(f"      {text[:100]}")

# InvokeModel, the documented fallback. Uses the runtime model ID; a mantle ID is
# not resolvable here (00-foundations/04).
print("\nguardrailIdentifier on InvokeModel:")
br = boto3.client("bedrock-runtime", region_name=REGION)
invoke_rows = []
for label, model_id, payload in [
    ("InvokeModel (Claude)", "us.anthropic.claude-opus-5",
     {"anthropic_version": "bedrock-2023-05-31", "max_tokens": 300,
      "messages": [{"role": "user", "content": DENIED}]}),
]:
    for guarded in (False, True):
        kwargs = {"modelId": model_id, "body": json.dumps(payload),
                  "contentType": "application/json"}
        if guarded:
            kwargs["guardrailIdentifier"] = GUARDRAIL_ID
            kwargs["guardrailVersion"] = GUARDRAIL_VERSION
        try:
            reply = br.invoke_model(**kwargs)
            parsed = json.loads(reply["body"].read())
            text = "".join(b.get("text", "") for b in (parsed.get("content") or [])
                           if isinstance(b, dict))
            stop = parsed.get("stop_reason")
            # Two independent signals, and it matters which one fired: InvokeModel
            # substitutes the guardrail's blocked message into the output and leaves
            # stop_reason at "end_turn", so naming stop_reason beside a verdict the
            # message produced reads like the verdict came from stop_reason.
            by_message = BLOCK_MARK in text
            by_stop = stop == "guardrail_intervened"
            hit = by_message or by_stop
            signal = (", ".join(s for s, on in
                                (("blocked message substituted", by_message),
                                 ("stop_reason", by_stop)) if on) or "neither")
            state = ("ENFORCED" if hit else "answered, NOT blocked") if guarded \
                else ("blocked without a guardrail?!" if hit else "answered")
            invoke_rows.append({"guarded": guarded, "enforced": hit, "status": 200})
            print(f"  {label} guardrail={guarded!s:<5} -> 200, {state} "
                  f"[signal: {signal}; stop_reason={stop!r}]")
        except Exception as exc:  # noqa: BLE001 - the error IS the measurement
            invoke_rows.append({"guarded": guarded, "enforced": False, "status": -1})
            print(f"  {label} guardrail={guarded!s:<5} -> {type(exc).__name__}: "
                  f"{str(exc)[:110]}")

# Derive the sentence, do not write it.
ignored_body = [r["surface"] for r in body_rows
                if r["status"] == 200 and not r["enforced"]]
refused_body = [r["surface"] for r in body_rows if r["status"] != 200]
invoke_enforces = any(r["enforced"] for r in invoke_rows if r["guarded"])
print(f"\n=> guardrailConfig in the body: SILENTLY IGNORED on {ignored_body or 'none'}")
print(f"=> guardrailConfig in the body: REFUSED on            {refused_body or 'none'}")
print(f"=> InvokeModel with guardrailIdentifier enforces:     {invoke_enforces}")
print("   Every row of the §9b table is now measured by this notebook.")


guardrailConfig in the request BODY (not a header):


  runtime Chat Completions   200 and SILENTLY IGNORED


  runtime Responses          400 REFUSED — safe, fails in development
      {"error":{"message":"Unknown parameter: 'guardrailConfig'.","type":"invalid_request_error","param":"


  runtime Messages           400 REFUSED — safe, fails in development
      {"type":"error","error":{"type":"invalid_request_error","message":"guardrailConfig: Extra inputs are


  mantle Chat Completions    200 and SILENTLY IGNORED

guardrailIdentifier on InvokeModel:


  InvokeModel (Claude) guardrail=False -> 200, answered [signal: neither; stop_reason='end_turn']


  InvokeModel (Claude) guardrail=True  -> 200, ENFORCED [signal: blocked message substituted; stop_reason='end_turn']

=> guardrailConfig in the body: SILENTLY IGNORED on ['runtime Chat Completions', 'mantle Chat Completions']
=> guardrailConfig in the body: REFUSED on            ['runtime Responses', 'runtime Messages']
=> InvokeModel with guardrailIdentifier enforces:     True
   Every row of the §9b table is now measured by this notebook.


In [25]:
# Delete the throwaway guardrail. Only the one this notebook created.
control.delete_guardrail(guardrailIdentifier=GUARDRAIL_ID)
# Paginate. An unpaginated list_guardrails() in an account with more guardrails than
# one page would report a still-existing throwaway guardrail as deleted.
remaining = []
_token = None
while True:
    _kwargs = {"nextToken": _token} if _token else {}
    _page = control.list_guardrails(**_kwargs)
    remaining += [g["name"] for g in _page.get("guardrails", [])]
    _token = _page.get("nextToken")
    if not _token:
        break
# Derived from the guardrail this notebook created. The hardcoded string said
# "mantle-samples-hardening-demo" while cell 34 created
# "bedrock-samples-hardening-demo", so a reader auditing their account for leftovers
# searched for a name that never existed.
# Check the name is actually absent rather than printing a count that says nothing
# about whether the delete worked.
gone = GUARDRAIL_NAME not in remaining
safe_print(f"deleted {GUARDRAIL_NAME}: {gone} | guardrails left in account:",
           len(remaining))
if not gone:
    safe_print("!! it is still listed -- delete it by hand before you forget")

deleted bedrock-samples-hardening-demo: True | guardrails left in account: 2


## 10. Pre-launch checklist

Run through this before your first real traffic.

In [26]:
# The checklist. Seventeen items; the notebook can DERIVE six of them from what it
# just did, and the other eleven are guidance for your deployment.
#
# Read the history, because it is the point of this cell. Version one carried a
# hardcoded `True` beside every item and summed the literals -- "17/17 controls
# implemented". Version two replaced those with expressions, which looked derived and
# was not: `529 in TRANSIENT` reads a literal set, `bool(SERVER_FAULT_TEXT)` reads a
# literal tuple, `hasattr(tokens, "get")` reads a method name. All six were True for
# any code at all, including code with the retry policy deleted. One of them was even
# inverted: a non-empty SERVER_FAULT_TEXT is precisely why this client DOES retry
# some 4xx, so it was ticking "fail fast on other 4xx" with the evidence against it.
#
# A tick has to be able to fail. So each one below is either a truth table over the
# real decision function or a measurement taken during this run, and the cell asserts
# that at least one formulation could have come out False.
CHECKLIST = [
    "Hardened client mints tokens in-process, TTL <= 15 min",
    "No long-term API keys anywhere in the deployment",
    "Retry with exponential backoff on 429 and 5xx",
    "Fail fast (no retries) on other 4xx",
    "Client-side timeout on every call",
    "Concurrency ramps over minutes, not seconds",
    "store=False unless server-side state is required",
    "data_retention mode chosen deliberately (none for regulated data)",
    "Every workload runs under a tagged Project",
    "Dashboards point at AWS/BedrockMantle, not AWS/Bedrock",
    "5xx tracked client-side (no server metric exists)",
    "Structured output parsed leniently and validated",
    "Sampling params gated per model",
    "service_tier gated per model",
    "Region pre-flight check for every required model",
    "Quota escalation path known (Support case, not Service Quotas)",
    "Guardrails applied via Converse or ApplyGuardrail, not a header",
]

# --- the evidence, computed now -------------------------------------------------
# Retries: ask the predicate what it does with each transient status, and with a
# permanent 400. All five answers must be right, so breaking the policy flips this.
# (status, label) -> (what should_retry decides, what it MUST decide). The EXPECTED
# column is written out, not derived from TRANSIENT. Deriving it was a real defect:
# the filter read `k[0] in TRANSIENT`, the same literal set should_retry consults, so
# deleting 529 from TRANSIENT removed the retry AND the row that would have caught
# it -- the summary still printed 6/6 while the line eight rows below read
# `529 overloaded -> retry=False`. An expectation that moves with the code is not an
# expectation. 502 and 504 are here too; nothing checked them before.
retry_table = {
    (429, "throttled"): (should_retry(429, {}), True),
    (502, "bad gateway"): (should_retry(502, {}), True),
    (503, "unavailable"): (should_retry(503, {}), True),
    (504, "gateway timeout"): (should_retry(504, {}), True),
    (529, "overloaded"): (should_retry(529, {}), True),
    (400, "server fault in body"): (should_retry(
        400, {"error": {"message": "Bad request"}, "details": "internal server error"}
    ), True),
    (400, "genuine bad parameter"): (should_retry(
        400, {"error": {"message": "Invalid 'max_output_tokens': below minimum"}}
    ), False),
    (404, "not found"): (should_retry(404, {}), False),
    (200, "success"): (should_retry(200, {}), False),
}
retry_mismatches = [k for k, (got, want) in retry_table.items() if got != want]
retries_transient = not [k for k, (got, want) in retry_table.items()
                         if want and got != want]
fails_fast_on_4xx = not [k for k, (got, want) in retry_table.items()
                         if not want and got != want]

# And the observed run: the permanent-400 call in section 2 must have made exactly
# one attempt. CALL_LOG is what the client recorded, not what the source looks like.
fail_fast_calls = [c for c in CALL_LOG if c["statuses"] and c["statuses"][-1] == 400]
observed_no_retry = bool(fail_fast_calls) and all(
    len(c["statuses"]) == 1 and c["sleeps"] == 0 for c in fail_fast_calls
)

# Timeouts. Reading CALL_LOG's recorded value alone was not enough: it records
# resilient_call's PARAMETER, so replacing `open_https(req, timeout=timeout)` with a
# bare urlopen leaves the log saying 60 and the tick still passes. So prove the
# timeout reaches the socket by asking for one that cannot be met; -1 is what
# resilient_call returns when the transport gives up.
recorded_timeouts = bool(CALL_LOG) and all(
    isinstance(c["timeout"], (int, float)) and 0 < c["timeout"] <= 300
    for c in CALL_LOG
)
_t0 = time.perf_counter()
_tiny_code, _ = resilient_call(
    f"{PREFIX}/responses",
    {"model": MODEL, "input": "Reply OK", "max_output_tokens": 16},
    attempts=1, timeout=0.001,
)
_tiny_elapsed = time.perf_counter() - _t0
timeout_enforced = _tiny_code == -1 and _tiny_elapsed < 10
timeouts_everywhere = recorded_timeouts and timeout_enforced

# Tokens. `tokens` is the 15-minute TokenProvider from section 1, used by
# resilient_call and by ProductionClient. Be honest about the scope: the shared
# `post()` helper and `send()` in section 9b mint through provide_token() with NO
# expiry argument, which is the ~12 h default this notebook's section 1 documents.
# So this tick is about the provider the hardened client uses, not about every call
# on the page, and the item is worded that way.
ttl_ok = tokens.ttl <= timedelta(minutes=15) and tokens.mints >= 1

# 5xx counter: feed a throwaway recorder a 500 and check the number moves. The old
# version asked whether the dict had a "server_error" key, which a literal always has.
# `metrics.counts["server_error"] == 0` would be asserting that this RUN saw no 5xx,
# which says nothing about whether 5xx is tracked. Ask the recorder instead.
_probe = CallMetrics()
_probe.record(503, 0.1)
_probe.record(200, 0.1)
_probe.record(429, 0.1)
tracks_5xx = (_probe.counts["server_error"] == 1 and _probe.counts["ok"] == 1
              and _probe.counts["throttled"] == 1)

# Lenient parsing: run it on the malformed payloads from section 7 and require both
# that it recovers the object AND that validation rejects an off-schema value.
# Two halves, both required. Lenient parsing must recover an object that
# json.loads refuses, AND the required-key check that safe_structured() applies must
# actually reject an object that is missing one. SCHEMA has no enum, so the earlier
# version of this line raised KeyError on properties.language.enum -- an evidence
# expression that crashes is worse than one that is merely trivial.
_trailing = '{"language":"Go","typed":true}\nextra text'
try:
    json.loads(_trailing)
    _strict_refused = False
except json.JSONDecodeError:
    _strict_refused = True
_recovered = parse_json_lenient(_trailing)
_incomplete = parse_json_lenient('{"language":"Go"}')
_missing = set(SCHEMA["required"]) - set(_incomplete)
lenient_and_validated = (
    _strict_refused
    and isinstance(_recovered, dict)
    and _recovered.get("language") == "Go"
    and _missing == {"typed"}
)

EVIDENCE = {
    "Retry with exponential backoff on 429 and 5xx": retries_transient,
    "Fail fast (no retries) on other 4xx": fails_fast_on_4xx and observed_no_retry,
    "Client-side timeout on every call": timeouts_everywhere,
    "Hardened client mints tokens in-process, TTL <= 15 min": ttl_ok,
    "5xx tracked client-side (no server metric exists)": tracks_5xx,
    "Structured output parsed leniently and validated": lenient_and_validated,
}

width = max(len(item) for item in CHECKLIST)
checked = derived = 0
for item in CHECKLIST:
    if item in EVIDENCE:
        passed = bool(EVIDENCE[item])
        derived += 1
        checked += passed
        mark, note = ("x", "checked here") if passed else ("!", "CHECKED AND FAILING")
    else:
        mark, note = "-", "guidance"
    print(f"  [{mark}] {item:{width}}  {note}")
print(
    f"\n{checked}/{derived} of the controls this notebook can verify are in place; "
    f"the remaining {len(CHECKLIST) - derived} are guidance for your deployment."
)
print("  [x] verified in this run   [!] verified and FAILING   [-] not verifiable here")

# The retry truth table, printed, because "6/6" is only meaningful if you can see
# what was asked.
print("\nretry decisions behind the two retry ticks (expected is written out, not "
      "derived from TRANSIENT):")
for (status, label), (got, want) in retry_table.items():
    mark = "ok" if got == want else "MISMATCH"
    print(f"  {status} {label:<22} retry={got!s:<5} expected={want!s:<5} {mark}")
if retry_mismatches:
    print(f"  !! {len(retry_mismatches)} decision(s) disagree with the policy this "
          f"notebook documents: {retry_mismatches}")
print(f"  observed: {len(fail_fast_calls)} permanent-400 call(s), "
      f"{sum(len(c['statuses']) for c in fail_fast_calls)} attempt(s) total")
print(f"  timeout reaches the socket: a 0.001s deadline returned "
      f"{_tiny_code} after {_tiny_elapsed:.2f}s")

# A tick that cannot fail is not a check. Prove the mechanism has teeth by asking
# what the display would do with a broken policy, without changing the real one.
_broken = dict(EVIDENCE, **{"Fail fast (no retries) on other 4xx": False})
print(f"\nsanity: with one control broken the summary would read "
      f"{sum(bool(v) for v in _broken.values())}/{len(_broken)}, and that row "
      f"would print [!]")


  [x] Hardened client mints tokens in-process, TTL <= 15 min             checked here
  [-] No long-term API keys anywhere in the deployment                   guidance
  [x] Retry with exponential backoff on 429 and 5xx                      checked here
  [x] Fail fast (no retries) on other 4xx                                checked here
  [x] Client-side timeout on every call                                  checked here
  [-] Concurrency ramps over minutes, not seconds                        guidance
  [-] store=False unless server-side state is required                   guidance
  [-] data_retention mode chosen deliberately (none for regulated data)  guidance
  [-] Every workload runs under a tagged Project                         guidance
  [-] Dashboards point at AWS/BedrockMantle, not AWS/Bedrock             guidance
  [x] 5xx tracked client-side (no server metric exists)                  checked here
  [x] Structured output parsed leniently and validated                   check

In [27]:
code, archived = post(
    f"/v1/organization/projects/{project_id}/archive", {}, region=REGION
)
print("cleaned up demo project:", code, archived.get("status"))

cleaned up demo project: 200 archived


## Gotchas — production on bedrock-mantle

| Gotcha | Detail |
|---|---|
| Token lifetime | ≤12 h, **not refreshable**, Region-pinned — mint and cache |
| No RPM quota | Token-based throttling only; most models have no published TPM |
| Quota increases | AWS Support case, **not** the Service Quotas console |
| Wrong path may stall | Always set a client timeout; bound your retries |
| `store` defaults to true | 30-day retention unless you opt out per request |
| Retention gating | A model can be `unavailable` under your retention mode |
| CloudWatch namespace | `AWS/BedrockMantle` — the wrong one shows zero errors |
| No 5xx metric | Track server errors from client-side telemetry |
| CloudTrail | Inference = data events (opt-in, extra cost) |
| Short-term key minting | Never logged — client-side by design |
| 200 ≠ correct | Truncation, trailing characters, ignored constraints all give 200 |
| Per-model params | Sampling and tier support vary — gate them |
| No CRIS / PT / batch | Cross-Region, Provisioned Throughput and batch are runtime-only |
| **No guardrails on mantle** | Not a parameter here: every mantle surface accepts the `X-Amzn-Bedrock-Guardrail*` headers and ignores them. On `bedrock-runtime` it is **per API** — §9b measures Chat Completions and Messages enforcing the header and Responses ignoring it — so do not read "OpenAI-shaped API" as "ignores guardrails". Converse and `ApplyGuardrail` enforce everywhere |
| Routing, not just prefixes | A client must resolve the API surface as well as the path, or it 400s on every Claude and Chat-Completions-only model |

## Where next
- `01-choosing-a-model-and-api.ipynb` — the live capability survey
- `02-migrating-from-openai.ipynb` — porting an existing codebase
- `../00-foundations/` — the underlying mechanics in depth